In [ ]:
# Run python script
import subprocess
import sys

args = ['hello']
kwargs = {'--n':'5'}
subprocess.run(
    [sys.executable, './delayed_message.py', 'Hello', '--n', '2'],
    check=True,
)

Hello


CompletedProcess(args=['/home/bashar/Study/Projects/AIVoiceAssistant/venv/bin/python', './delayed_message.py', 'Hello', '--n', '2'], returncode=0)

In [1]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage, AIMessage
from langchain_core.tools import tool, Tool
from pydantic import BaseModel


import subprocess
import sys 

In [ ]:

class ExecutePythonScriptInput(BaseModel):
    path: str
    args: list[str] | None = None
    kwargs: dict[str, str | int | float] | None = None


@tool(args_schema=ExecutePythonScriptInput)
def execute_python_script(
    path: str,
    args: list[str] | None = None,
    kwargs: dict[str, str | int | float] | None = None,
) -> str:
    """Run a Python script with positional args and named CLI options.

    Args:
        path: Path to the Python script.
        args: Positional command-line arguments.
        kwargs: Options with values, e.g. {"n": 3} becomes --n 3.
            Keys may include their leading dashes, e.g. --n or -n.
    """
    command = [sys.executable, path, *[str(arg) for arg in (args or [])]]
    for name, value in (kwargs or {}).items():
        option = name if name.startswith("-") else f"--{name}"
        command.extend([option, str(value)])
    result = subprocess.run(
        command,
        check=True,
        capture_output=True,
        text=True,
    )
    return result.stdout


@tool
def write_python_script(path: str , content: str):
    "a function to write a python script by specifyign the path and the content"
    with open(path, 'w') as f:
        f.write(content)

In [ ]:
chat = ChatOllama(
    model='qwen3.5:4b',
    temperature=0,
    max_tokens=256,
)

SystemPrompt = '''
You are a helpful assistant, the user will ask you to perform some task, you should find the appropriate tool to run the task.
### TOOLS:
- list_python_scripts: is used to list available scripts
- write_python_script: is used to write a specific python script to a specific directory.
- execute_python_script: is used to run a pre-defined python script
'''
tools = [execute_python_script, write_python_script]

messages = [HumanMessage(content='run a python script to print the word Hell after 3 seconds')]
messages_write_python_script = [HumanMessage(content='write a python script that takes two args and returns the sum, put the file in agent_generated_scripts directory and namd it proprely')]

def llm_chat(messages:list, tools: list | None = None) -> list: 
    tools_dict = {tool.name: tool for tool in tools}
    result = chat.bind_tools(tools).invoke([SystemMessage(content=SystemPrompt)] + messages)
    messages.append(result)
    if result.content:
        print(f"[AGENT]: {result.content}")
    if result.content == '' and result.tool_calls:
        for call in result.tool_calls:
            print(f"[TOOL CALL]: {call}")
            tool_call = tools_dict[call['name']]
            tool_result = tool_call.invoke(call)
            print(f"[OBS]: {tool_result.content}")
            messages.append(tool_result)
    return messages

In [40]:
result = llm_chat(messages_write_python_script, tools)

[TOOL CALL]: {'name': 'write_python_script', 'args': {'path': 'agent_generated_scripts/add_two_numbers.py', 'content': '#!/usr/bin/env python3\nimport sys\n\ndef add_two_numbers(a, b):\n    """Add two numbers and return the sum."""\n    return a + b\n\nif __name__ == "__main__":\n    if len(sys.argv) != 3:\n        print("Usage: python add_two_numbers.py <number1> <number2>")\n        sys.exit(1)\n    \n    try:\n        num1 = float(sys.argv[1])\n        num2 = float(sys.argv[2])\n        result = add_two_numbers(num1, num2)\n        print(f"Sum of {num1} and {num2} is: {result}")\n    except ValueError:\n        print("Error: Please provide two valid numbers as arguments.")\n        sys.exit(1)\n'}, 'id': 'bea4e630-8c0b-46c1-a6c8-82d0576ca4c3', 'type': 'tool_call'}
[OBS]: null


#### OKAY, we have a primitive (but working) llm agent, now let's add the ASR model:

In [ ]:
from qwen_asr import Qwen3ASRModel
import torch

model = Qwen3ASRModel.from_pretrained(
    "Qwen/Qwen3-ASR-0.6B",
    dtype=torch.bfloat16,
    device_map="cuda:0",
    # attn_implementation="flash_attention_2",
    max_inference_batch_size=32, # Batch size limit for inference. -1 means unlimited. Smaller values can help avoid OOM.
    max_new_tokens=256, # Maximum number of tokens to generate. Set a larger value for long audio input.
)



/home/bashar/Study/Projects/AIVoiceAssistant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


In [17]:
%time
results = model.transcribe(
    audio="recording.wav",
    language="English", # set "English" to force the language
)
print(results)

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


CPU times: user 5 μs, sys: 0 ns, total: 5 μs
Wall time: 9.3 μs
[ASRTranscription(language='English', text='Hello there. This is a test.', time_stamps=None)]


### Combine audio -> ASR -> LLM -> Output